In [2]:
!pip install ultralytics sahi albumentations opencv-python-headless -q

import os, shutil, random, math, cv2
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import albumentations as A

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 29.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [6]:
BASE     = Path("/kaggle/input/datasets/banuprasadb/visdrone-dataset")
TRAIN_IMGS = BASE / "/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-train/images"
TRAIN_LBLS = BASE / "/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-train/labels"
VAL_IMGS   = BASE / "/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-val/images"
VAL_LBLS   = BASE / "/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-val/labels"

for p in [TRAIN_IMGS, TRAIN_LBLS, VAL_IMGS, VAL_LBLS]:
    count = len(list(p.glob("*")))
    print(f"{'OK' if count > 0 else 'MISSING'} | {count:>5} files | {p}")

OK |  6471 files | /kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-train/images
OK |  6471 files | /kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-train/labels
OK |   548 files | /kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-val/images
OK |   548 files | /kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-val/labels


In [4]:
from collections import Counter

# Count total annotations
total_boxes = 0
class_counts = Counter()

for lbl_file in TRAIN_LBLS.glob("*.txt"):
    with open(lbl_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:        # valid YOLO line
                cls = int(parts[0])
                class_counts[cls] += 1
                total_boxes += 1

print(f"Total annotations in training set: {total_boxes}")
print("Annotations per class:")
for cls, cnt in sorted(class_counts.items()):
    print(f"  Class {cls}: {cnt}")

KeyboardInterrupt: 

In [4]:
import cv2
from pathlib import Path
import numpy as np

# folder containing your images
img_folder = TRAIN_IMGS

brightness_values = []

for img_path in img_folder.glob("*.jpg"):
    img = cv2.imread(str(img_path))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # convert to grayscale
    avg_brightness = np.mean(gray)  # average pixel intensity
    brightness_values.append(avg_brightness)

overall_avg = np.mean(brightness_values)
print(f"Average brightness over all images: {overall_avg:.2f}")

KeyboardInterrupt: 

In [7]:
TARGET_SIZE = 640
MIN_PX = 2 / TARGET_SIZE   # 2px threshold in normalised coords

def clean_label_file(lbl_path: Path, out_path: Path):
    lines = lbl_path.read_text().strip().splitlines()
    kept = []
    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            continue
        cls, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
        if w < MIN_PX or h < MIN_PX:   # too tiny
            continue
        kept.append(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text("\n".join(kept))
    return len(kept)

CLEAN_TRAIN_IMGS = Path("/kaggle/working/clean/train/images")
CLEAN_TRAIN_LBLS = Path("/kaggle/working/clean/train/labels")
CLEAN_VAL_IMGS   = Path("/kaggle/working/clean/val/images")
CLEAN_VAL_LBLS   = Path("/kaggle/working/clean/val/labels")

for d in [CLEAN_TRAIN_IMGS, CLEAN_TRAIN_LBLS, CLEAN_VAL_IMGS, CLEAN_VAL_LBLS]:
    d.mkdir(parents=True, exist_ok=True)

# --- clean train ---
total_kept = 0
for img_path in tqdm(sorted(TRAIN_IMGS.glob("*.jpg")), desc="Cleaning train"):
    lbl_path = TRAIN_LBLS / (img_path.stem + ".txt")
    if not lbl_path.exists():
        continue
    out_lbl = CLEAN_TRAIN_LBLS / lbl_path.name
    kept = clean_label_file(lbl_path, out_lbl)
    if kept > 0:
        shutil.copy(img_path, CLEAN_TRAIN_IMGS / img_path.name)
        total_kept += kept

print(f"Train: kept {total_kept} annotations")

# --- clean val ---
for img_path in tqdm(sorted(VAL_IMGS.glob("*.jpg")), desc="Cleaning val"):
    lbl_path = VAL_LBLS / (img_path.stem + ".txt")
    if not lbl_path.exists():
        continue
    out_lbl = CLEAN_VAL_LBLS / lbl_path.name
    kept = clean_label_file(lbl_path, out_lbl)
    if kept > 0:
        shutil.copy(img_path, CLEAN_VAL_IMGS / img_path.name)

Cleaning train:   0%|          | 0/6471 [00:00<?, ?it/s]

Train: kept 335288 annotations


Cleaning val:   0%|          | 0/548 [00:00<?, ?it/s]

In [6]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

CLAHE_IMGS = Path("/kaggle/working/clahe/train/images")
CLAHE_IMGS.mkdir(parents=True, exist_ok=True)

for img_path in tqdm(sorted(CLEAN_TRAIN_IMGS.glob("*.jpg")), desc="CLAHE"):
    img = cv2.imread(str(img_path))
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_clahe = clahe.apply(l)
    merged = cv2.merge([l_clahe, a, b])
    result = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
    cv2.imwrite(str(CLAHE_IMGS / img_path.name), result)

CLAHE_LBLS = Path("/kaggle/working/clahe/train/labels")
CLAHE_LBLS.mkdir(parents=True, exist_ok=True)
for lbl in CLEAN_TRAIN_LBLS.glob("*.txt"):
    shutil.copy(lbl, CLAHE_LBLS / lbl.name)

print("CLAHE done:", len(list(CLAHE_IMGS.glob("*.jpg"))), "images")

CLAHE:   0%|          | 0/6471 [00:00<?, ?it/s]

CLAHE done: 6471 images


In [7]:
!rm -rf /kaggle/working/clean/train

In [7]:
#Check brightness after CLAHE
import cv2
from pathlib import Path
import numpy as np

# folder containing your images
img_folder = CLAHE_IMGS

brightness_values = []

for img_path in img_folder.glob("*.jpg"):
    img = cv2.imread(str(img_path))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # convert to grayscale
    avg_brightness = np.mean(gray)  # average pixel intensity
    brightness_values.append(avg_brightness)

overall_avg = np.mean(brightness_values)
print(f"Average brightness over all images: {overall_avg:.2f}")

KeyboardInterrupt: 

In [8]:

import cv2
import numpy as np
import albumentations as A
from pathlib import Path
from tqdm.auto import tqdm
import random

AUG_IMGS = Path("/kaggle/working/augmented/images")
AUG_LBLS = Path("/kaggle/working/augmented/labels")
AUG_IMGS.mkdir(parents=True, exist_ok=True)
AUG_LBLS.mkdir(parents=True, exist_ok=True)

TILE_SIZE   = 640
OVERLAP     = 0.15
MIN_VIS     = 0.25
MIN_PX_NORM = 2 / TILE_SIZE
N_AUG       = 0        
SKIP_EMPTY  = True     
JPEG_Q      = 75       

transform = A.Compose([
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=40, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.RandomShadow(shadow_roi=(0,0,1,1), num_shadows_lower=1,
                   num_shadows_upper=3, shadow_dimension=5, p=0.4),
    A.GaussNoise(var_limit=(5,25), p=0.3),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.0),
    A.RandomRotate90(p=0.7),
    A.ImageCompression(quality_lower=75, quality_upper=100, p=0.3),
],
bbox_params=A.BboxParams(
    format="yolo",
    label_fields=["class_labels"],
    min_visibility=0.3,
    clip=True
))

def load_yolo(lbl_path):
    labels, boxes = [], []
    if not lbl_path.exists():
        return labels, boxes
    for line in lbl_path.read_text().strip().splitlines():
        p = line.split()
        if len(p) == 5:
            labels.append(int(p[0]))
            boxes.append(list(map(float, p[1:])))
    return labels, boxes

def save_yolo(lbl_path, labels, boxes):
    lines = [f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}"
             for c, (x, y, w, h) in zip(labels, boxes)]
    lbl_path.write_text("\n".join(lines))

def get_tile_boxes(boxes, labels, x1, y1, x2, y2, W, H):
    tile_w = x2 - x1
    tile_h = y2 - y1
    out_labels, out_boxes = [], []

    for cls, (xc, yc, bw, bh) in zip(labels, boxes):
        # to pixel
        bx1 = (xc - bw/2) * W;  bx2 = (xc + bw/2) * W
        by1 = (yc - bh/2) * H;  by2 = (yc + bh/2) * H

        # clip to tile
        cx1 = max(bx1, x1);  cx2 = min(bx2, x2)
        cy1 = max(by1, y1);  cy2 = min(by2, y2)
        if cx2 <= cx1 or cy2 <= cy1:
            continue

        # visibility
        orig_area    = max((bx2-bx1)*(by2-by1), 1e-6)
        clipped_area = (cx2-cx1)*(cy2-cy1)
        if clipped_area / orig_area < MIN_VIS:
            continue

        # normalise to tile
        nxc = ((cx1+cx2)/2 - x1) / tile_w
        nyc = ((cy1+cy2)/2 - y1) / tile_h
        nbw = (cx2-cx1) / tile_w
        nbh = (cy2-cy1) / tile_h

        if nbw < MIN_PX_NORM or nbh < MIN_PX_NORM:
            continue

        out_labels.append(cls)
        out_boxes.append([nxc, nyc, nbw, nbh])

    return out_labels, out_boxes

def write_tile(img_tile, labels, boxes, name):
    if SKIP_EMPTY and not boxes:
        return False
    cv2.imwrite(
        str(AUG_IMGS / (name + ".jpg")),
        img_tile,
        [cv2.IMWRITE_JPEG_QUALITY, JPEG_Q]
    )
    save_yolo(AUG_LBLS / (name + ".txt"), labels, boxes)
    return True


img_paths   = sorted(CLAHE_IMGS.glob("*.jpg"))
total_written = 0
step = int(TILE_SIZE * (1 - OVERLAP))

for img_path in tqdm(img_paths, desc="Tile + Augment"):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    H, W = img.shape[:2]

    labels, boxes = load_yolo(CLAHE_LBLS / (img_path.stem + ".txt"))

    # Tile origin points
    xs = sorted(set(list(range(0, max(1,W-TILE_SIZE), step)) + [max(0,W-TILE_SIZE)]))
    ys = sorted(set(list(range(0, max(1,H-TILE_SIZE), step)) + [max(0,H-TILE_SIZE)]))

    for ti, y1 in enumerate(ys):
        for tj, x1 in enumerate(xs):
            x2 = min(x1 + TILE_SIZE, W)
            y2 = min(y1 + TILE_SIZE, H)

            tile_labels, tile_boxes = get_tile_boxes(
                boxes, labels, x1, y1, x2, y2, W, H
            )

            if SKIP_EMPTY and not tile_boxes:
                continue

            tile_img = img[y1:y2, x1:x2]  
            img_rgb  = cv2.cvtColor(tile_img, cv2.COLOR_BGR2RGB)

            name = f"{img_path.stem}_t{ti}_{tj}"
            if write_tile(tile_img, tile_labels, tile_boxes, name):
                total_written += 1

            for aug_i in range(N_AUG):
                try:
                    result = transform(
                        image=img_rgb,
                        bboxes=tile_boxes,
                        class_labels=tile_labels
                    )
                except Exception:
                    continue
                if not result["bboxes"]:
                    continue
                aug_bgr = cv2.cvtColor(result["image"], cv2.COLOR_RGB2BGR)
                aug_name = f"{name}_a{aug_i}"
                if write_tile(aug_bgr, result["class_labels"],
                              list(result["bboxes"]), aug_name):
                    total_written += 1

    del img

print(f"\nTotal images written: {total_written:,}")
!df -h /kaggle/working
!du -sh /kaggle/working/augmented

/tmp/ipykernel_57/3116656182.py:24: UserWarning: Argument(s) 'num_shadows_lower, num_shadows_upper' are not valid for transform RandomShadow
  A.RandomShadow(shadow_roi=(0,0,1,1), num_shadows_lower=1,
/tmp/ipykernel_57/3116656182.py:26: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5,25), p=0.3),
/tmp/ipykernel_57/3116656182.py:30: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, quality_upper=100, p=0.3),


Tile + Augment:   0%|          | 0/6471 [00:00<?, ?it/s]


Total images written: 39,927
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  6.9G   13G  36% /kaggle/working
3.3G	/kaggle/working/augmented


In [4]:
# Final dataset YAML
from pathlib import Path

yaml_content = f"""
path: /kaggle/working
train: augmented/images
val: clean/val/images
test: /kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-test-dev/images

nc: 10
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

yaml_path = Path("/kaggle/working/visdrone_processed.yaml")
yaml_path.write_text(yaml_content.strip())

# Sanity check
train_imgs = len(list(Path("/kaggle/working/augmented/images").glob("*.jpg")))
val_imgs   = len(list(Path("/kaggle/working/clean/val/images").glob("*.jpg")))
print(f"Train images : {train_imgs:,}")
print(f"Val images   : {val_imgs:,}")
print(f"\nYAML written to {yaml_path}")
print(yaml_path.read_text())

Train images : 0
Val images   : 0

YAML written to /kaggle/working/visdrone_processed.yaml
path: /kaggle/working
train: augmented/images
val: clean/val/images
test: /kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-test-dev/images

nc: 10
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor


In [10]:
#Train YOLOv8
from ultralytics import YOLO
import numpy as np

model = YOLO("/kaggle/input/models/tausifulislam9786/yolov8/pytorch/default/1/epoch20.pt")
model.train(
    data="/kaggle/working/visdrone_processed.yaml",
    epochs=5,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    optimizer="AdamW",
    lr0=1e-3,
    lrf=0.01,
    warmup_epochs=3,
    cls=0.5,            
    box=7.5,            
    project="/kaggle/working/second_runs",
    name="visdrone_yolov8m_next25",
    exist_ok=True,
    save=True,
    plots=True,
    save_period=10,     
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/visdrone_processed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bd93aa92cc0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [11]:
# for saving
import shutil
from pathlib import Path

shutil.make_archive(
    "/kaggle/working/visdrone_second_weights",   # output zip name
    "zip",
    "/kaggle/working/second_runs/visdrone_yolov8m_next25/weights"
)

print("Saved.zip")

Saved.zip


In [8]:
# Cell 10: Evaluate on test set and count
from ultralytics import YOLO
import cv2
import time
from pathlib import Path

model = YOLO("/kaggle/input/models/tausifulislam9786/yolov8/pytorch/default/1/epoch20.pt")

HUMAN_CLASSES = {0, 1}  # pedestrian, people
CAR_CLASS     = {3}      # car

TEST_DIR = Path("/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-test-dev/images")
OUT_DIR  = Path("/kaggle/working/test_outputs/counted")
OUT_DIR.mkdir(parents=True, exist_ok=True)

start = time.time()


metrics = model.val(
    data="/kaggle/working/visdrone_processed.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    save_json=True,
    plots=True
)

total_humans = 0
total_cars   = 0
img_paths    = sorted(TEST_DIR.glob("*.jpg"))

for img_path in img_paths:
    img     = cv2.imread(str(img_path))
    results = model(img, imgsz=640, conf=0.25, verbose=False)[0]

    human_count = 0
    car_count   = 0

    for box in results.boxes:
        cls          = int(box.cls)
        conf         = float(box.conf)
        x1,y1,x2,y2 = map(int, box.xyxy[0])

        if cls in HUMAN_CLASSES:
            human_count += 1
            cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(img, f"human {conf:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,255,0), 1)

        elif cls in CAR_CLASS:
            car_count += 1
            cv2.rectangle(img, (x1,y1), (x2,y2), (255,0,0), 2)
            cv2.putText(img, f"car {conf:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,0,0), 1)

    cv2.putText(img, f"Humans: {human_count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2)
    cv2.putText(img, f"Cars: {car_count}", (10, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,0,0), 2)

    cv2.imwrite(str(OUT_DIR / img_path.name), img)

    total_humans += human_count
    total_cars   += car_count

end = time.time()

print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")
print(f"mAP50      : {metrics.box.map50:.4f}")
print(f"mAP50-95   : {metrics.box.map:.4f}")

print("DETECTION COUNTS")
print("=" * 45)
print(f"Images processed : {len(img_paths)}")
print(f"Total humans     : {total_humans}")
print(f"Total cars       : {total_cars}")
print(f"Total time       : {end-start:.2f}s")
print(f"Annotated images saved to: {OUT_DIR}")

Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,129,454 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 31.0±11.1 MB/s, size: 227.8 KB)
val: Scanning /kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-test-dev/labels... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 196.5it/s 8.2s<0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-test-dev is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 3.6it/s 28.0s0.2s
                   all       1610      75102      0.434      0.311       0.28      0.158
            pedestrian       1197      21006      0.431      0.234      0.216     0.0873
                people        797       6376      0.468      0.126      0.

In [10]:
# for saving
import shutil
from pathlib import Path

shutil.make_archive(
    "/kaggle/working/inference",   # output zip name
    "zip",
    "/kaggle/working/test_outputs/counted"
)

print("Saved.zip")

KeyboardInterrupt: 